# Time-structure audit

This notebook inspects timestamps, sampling regularity, and run boundaries without changing the source data. It creates a run-level summary and an issue-level audit that can be reviewed before future-label engineering.

### 1. Imports, paths, and source fingerprint

**What this cell does:** Imports the analysis tools, defines repository-relative paths, and records the input file's SHA-256 checksum.  
**Why it matters for time-series ML:** Reproducible paths prevent analysis of the wrong file, while a checksum proves the audit is non-destructive.  
**What to understand from the result:** The exact input and output locations and the input fingerprint before analysis.

In [ ]:
from collections import defaultdict
from itertools import combinations
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_metrics.csv"
TIME_AUDIT_PATH = PROJECT_ROOT / "reports" / "time_structure_audit.csv"
RUN_SUMMARY_PATH = PROJECT_ROOT / "reports" / "run_summary.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Input dataset not found: {INPUT_PATH}")

TIME_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
input_hash_before = sha256_file(INPUT_PATH)

print(f"Input: {INPUT_PATH}")
print(f"Issue report: {TIME_AUDIT_PATH}")
print(f"Run summary: {RUN_SUMMARY_PATH}")
print(f"Input SHA-256 before audit: {input_hash_before}")

### 2. Load the dataset without modifying it

**What this cell does:** Reads the CSV into `raw_df` and validates the columns required for time analysis.  
**Why it matters for time-series ML:** Machine, run, and timestamp keys define independent sequences; missing keys would invalidate later checks.  
**What to understand from the result:** The source shape and confirmation that all required keys are present.

In [ ]:
raw_df = pd.read_csv(INPUT_PATH)
required_columns = {"machine_id", "run_id", "timestamp"}
missing_required = required_columns - set(raw_df.columns)
if missing_required:
    raise KeyError(f"Required time-structure columns are missing: {sorted(missing_required)}")

original_columns = raw_df.columns.tolist()
original_shape = raw_df.shape

print(f"Loaded shape: {raw_df.shape}")
print(f"Required columns present: {sorted(required_columns)}")

### 3. Safely parse timestamps and sort an analysis copy

**What this cell does:** Creates a deep copy, parses timestamps as UTC with invalid values converted to `NaT`, preserves source-row order, and stably sorts the copy by machine, run, and timestamp.  
**Why it matters for time-series ML:** Interval and future-window calculations require chronological order, but sorting the original would hide whether the delivered file itself went backward.  
**What to understand from the result:** How many timestamps parsed successfully and that only `sorted_df`, never `raw_df`, is sorted.

In [ ]:
analysis_df = raw_df.copy(deep=True)
analysis_df["_source_row"] = np.arange(len(analysis_df))
analysis_df["_timestamp_raw"] = analysis_df["timestamp"].astype("string")
analysis_df["_parsed_timestamp"] = pd.to_datetime(
    analysis_df["timestamp"], errors="coerce", utc=True
)

# Backward movement must be measured in the original source order, before sorting.
analysis_df["_source_order_delta_seconds"] = (
    analysis_df.groupby(["machine_id", "run_id"], sort=False)["_parsed_timestamp"]
    .diff()
    .dt.total_seconds()
)

sorted_df = analysis_df.sort_values(
    ["machine_id", "run_id", "_parsed_timestamp", "_source_row"],
    kind="mergesort",
    na_position="last",
).copy()
sorted_df["sample_interval_seconds"] = (
    sorted_df.groupby(["machine_id", "run_id"], sort=False)["_parsed_timestamp"]
    .diff()
    .dt.total_seconds()
)

invalid_timestamp_count = int(analysis_df["_parsed_timestamp"].isna().sum())
print(f"Successfully parsed timestamps: {len(analysis_df) - invalid_timestamp_count:,}")
print(f"Invalid timestamps converted safely to NaT: {invalid_timestamp_count:,}")
print("Original DataFrame shape remains:", raw_df.shape)

### 4. Count machines, runs, and rows per sequence

**What this cell does:** Counts distinct machines and run IDs and displays row counts by machine and by machine/run pair.  
**Why it matters for time-series ML:** Sequence size and imbalance affect validation design, window availability, and how much evidence each machine contributes.  
**What to understand from the result:** How many independent entities and runs exist and whether some sequences are much shorter than others.

In [ ]:
machine_count = int(raw_df["machine_id"].nunique(dropna=True))
run_id_count = int(raw_df["run_id"].nunique(dropna=True))
machine_run_count = int(raw_df[["machine_id", "run_id"]].drop_duplicates().shape[0])

rows_per_machine = raw_df.groupby("machine_id", dropna=False).size().rename("row_count")
rows_per_run = raw_df.groupby(["machine_id", "run_id"], dropna=False).size().rename("row_count")

print(f"Machines found: {machine_count}")
print(f"Distinct run_id values: {run_id_count}")
print(f"Machine/run sequences: {machine_run_count}")
print("\nRows per machine:")
display(rows_per_machine.to_frame())
print("Rows per machine/run:")
display(rows_per_run.to_frame())

### 5. Define the normal interval, large-gap rule, and run statistics

**What this cell does:** Uses the median of all positive within-run intervals as the normal sampling interval and defines a large gap as strictly greater than **5 × that median**. It then calculates run boundaries, duration, and interval statistics.  
**Why it matters for time-series ML:** The median resists distortion from outages, and the 5× multiplier allows ordinary timing jitter while flagging interruptions large enough to break future windows.  
**What to understand from the result:** The typical cadence, the exact gap threshold, and each run's median, mean, minimum, maximum, and 95th-percentile interval.

In [ ]:
positive_intervals = sorted_df.loc[
    sorted_df["sample_interval_seconds"].gt(0), "sample_interval_seconds"
]
normal_sampling_interval_seconds = float(positive_intervals.median()) if len(positive_intervals) else np.nan
LARGE_GAP_MULTIPLIER = 5.0
large_gap_threshold_seconds = (
    normal_sampling_interval_seconds * LARGE_GAP_MULTIPLIER
    if np.isfinite(normal_sampling_interval_seconds)
    else np.nan
)

run_summary = (
    sorted_df.groupby(["machine_id", "run_id"], dropna=False)
    .agg(
        row_count=("_source_row", "size"),
        valid_timestamp_count=("_parsed_timestamp", "count"),
        first_timestamp=("_parsed_timestamp", "min"),
        last_timestamp=("_parsed_timestamp", "max"),
    )
    .reset_index()
)
run_summary["invalid_timestamp_count"] = run_summary["row_count"] - run_summary["valid_timestamp_count"]
run_summary["run_duration_seconds"] = (
    run_summary["last_timestamp"] - run_summary["first_timestamp"]
).dt.total_seconds()

interval_summary = (
    sorted_df.loc[sorted_df["sample_interval_seconds"].gt(0)]
    .groupby(["machine_id", "run_id"])["sample_interval_seconds"]
    .agg(
        interval_median_seconds="median",
        interval_mean_seconds="mean",
        interval_min_seconds="min",
        interval_max_seconds="max",
        interval_p95_seconds=lambda values: values.quantile(0.95),
    )
    .reset_index()
)
run_summary = run_summary.merge(interval_summary, on=["machine_id", "run_id"], how="left", validate="one_to_one")
run_summary["normal_sampling_interval_seconds"] = normal_sampling_interval_seconds
run_summary["large_gap_threshold_seconds"] = large_gap_threshold_seconds

print(f"Normal sampling interval (global median): {normal_sampling_interval_seconds:.3f} seconds")
print(f"Large-gap rule: interval > {LARGE_GAP_MULTIPLIER:g} × median = {large_gap_threshold_seconds:.3f} seconds")
display(run_summary)

### 6. Detect invalid, duplicated, backward, and gapped timestamps

**What this cell does:** Finds timestamp parse failures, duplicate timestamps inside a machine/run, backward steps in source order, and large gaps in chronological order.  
**Why it matters for time-series ML:** These conditions can create incorrect window lengths, duplicate observations, broken causal ordering, or labels that span periods with no measurements.  
**What to understand from the result:** The number of each row-level issue and which runs contain interruptions.

In [ ]:
issue_records = []
group_keys = ["machine_id", "run_id"]

def timestamp_text(value):
    return "" if pd.isna(value) else value.isoformat()

invalid_rows = analysis_df[analysis_df["_parsed_timestamp"].isna()]
for _, row in invalid_rows.iterrows():
    issue_records.append({
        "machine_id": row["machine_id"],
        "run_id": row["run_id"],
        "issue_type": "invalid_timestamp",
        "timestamp": row["_timestamp_raw"],
        "gap_seconds": np.nan,
        "severity": "error",
        "explanation": "Timestamp could not be parsed safely as UTC and cannot be placed on the time axis.",
    })

duplicate_all_mask = sorted_df.duplicated(group_keys + ["_parsed_timestamp"], keep=False) & sorted_df["_parsed_timestamp"].notna()
duplicate_extra_mask = sorted_df.duplicated(group_keys + ["_parsed_timestamp"], keep="first") & sorted_df["_parsed_timestamp"].notna()
for _, row in sorted_df.loc[duplicate_extra_mask].iterrows():
    issue_records.append({
        "machine_id": row["machine_id"],
        "run_id": row["run_id"],
        "issue_type": "duplicated_timestamp",
        "timestamp": timestamp_text(row["_parsed_timestamp"]),
        "gap_seconds": 0.0,
        "severity": "error",
        "explanation": "An additional row has the same timestamp inside this machine/run sequence.",
    })

backward_mask = analysis_df["_source_order_delta_seconds"].lt(0)
for _, row in analysis_df.loc[backward_mask].iterrows():
    issue_records.append({
        "machine_id": row["machine_id"],
        "run_id": row["run_id"],
        "issue_type": "timestamp_backward",
        "timestamp": timestamp_text(row["_parsed_timestamp"]),
        "gap_seconds": float(row["_source_order_delta_seconds"]),
        "severity": "error",
        "explanation": "Timestamp is earlier than the preceding row from the same machine/run in source order.",
    })

large_gap_mask = sorted_df["sample_interval_seconds"].gt(large_gap_threshold_seconds)
for _, row in sorted_df.loc[large_gap_mask].iterrows():
    issue_records.append({
        "machine_id": row["machine_id"],
        "run_id": row["run_id"],
        "issue_type": "large_gap",
        "timestamp": timestamp_text(row["_parsed_timestamp"]),
        "gap_seconds": float(row["sample_interval_seconds"]),
        "severity": "warning",
        "explanation": f"Interval exceeds the {LARGE_GAP_MULTIPLIER:g}× normal-interval threshold of {large_gap_threshold_seconds:.3f} seconds.",
    })

duplicate_counts = (
    sorted_df.assign(_is_duplicate_timestamp_row=duplicate_all_mask)
    .groupby(group_keys)["_is_duplicate_timestamp_row"].sum().astype(int)
)
backward_counts = (
    analysis_df.assign(_is_backward=backward_mask)
    .groupby(group_keys)["_is_backward"].sum().astype(int)
)
large_gap_stats = (
    sorted_df.assign(_is_large_gap=large_gap_mask)
    .groupby(group_keys)
    .agg(
        large_gap_count=("_is_large_gap", "sum"),
        maximum_gap_seconds=("sample_interval_seconds", "max"),
    )
)

print(f"Invalid timestamps: {len(invalid_rows)}")
print(f"Rows involved in duplicated timestamps: {int(duplicate_all_mask.sum())}")
print(f"Backward timestamp steps: {int(backward_mask.sum())}")
print(f"Large gaps: {int(large_gap_mask.sum())}")

### 7. Detect run overlaps and run IDs shared across machines

**What this cell does:** Compares run time ranges pairwise within each machine and checks whether any `run_id` belongs to multiple machines.  
**Why it matters for time-series ML:** Overlapping runs may mix concurrent collection sessions, while a run ID reused across machines can make grouping and leakage-safe splitting ambiguous.  
**What to understand from the result:** Whether run boundaries and run identifiers uniquely define independent sequences.

In [ ]:
overlap_counts = defaultdict(int)
overlap_pair_count = 0

for machine_id, machine_runs in run_summary.groupby("machine_id", dropna=False):
    valid_runs = machine_runs.dropna(subset=["first_timestamp", "last_timestamp"])
    for left, right in combinations(valid_runs.itertuples(index=False), 2):
        overlap_start = max(left.first_timestamp, right.first_timestamp)
        overlap_end = min(left.last_timestamp, right.last_timestamp)
        if overlap_start <= overlap_end:
            overlap_pair_count += 1
            overlap_seconds = float((overlap_end - overlap_start).total_seconds())
            overlap_counts[(machine_id, left.run_id)] += 1
            overlap_counts[(machine_id, right.run_id)] += 1
            issue_records.append({
                "machine_id": machine_id,
                "run_id": right.run_id,
                "issue_type": "overlapping_runs",
                "timestamp": timestamp_text(overlap_start),
                "gap_seconds": np.nan,
                "severity": "error",
                "explanation": f"Run overlaps run_id {left.run_id} on the same machine for {overlap_seconds:.3f} seconds.",
            })

run_machine_counts = raw_df.groupby("run_id", dropna=False)["machine_id"].nunique(dropna=False)
shared_run_ids = run_machine_counts[run_machine_counts.gt(1)].index.tolist()
shared_run_keys = set()
for run_id in shared_run_ids:
    matching_runs = run_summary[run_summary["run_id"].eq(run_id)]
    machine_list = matching_runs["machine_id"].astype(str).tolist()
    for row in matching_runs.itertuples(index=False):
        shared_run_keys.add((row.machine_id, row.run_id))
        issue_records.append({
            "machine_id": row.machine_id,
            "run_id": row.run_id,
            "issue_type": "run_id_on_multiple_machines",
            "timestamp": timestamp_text(row.first_timestamp),
            "gap_seconds": np.nan,
            "severity": "error",
            "explanation": f"This run_id appears on multiple machines: {machine_list}.",
        })

print(f"Overlapping run pairs on the same machine: {overlap_pair_count}")
print(f"Run IDs appearing on more than one machine: {len(shared_run_ids)}")
display(shared_run_ids)

### 8. Build run safety decisions and the issue table

**What this cell does:** Adds issue counts to each run, assigns review reasons, and marks a run safe only when it has valid ordered unique timestamps, no large gaps, no overlaps, and an unambiguous run ID.  
**Why it matters for time-series ML:** Future labels require continuous, correctly ordered histories; an explicit safety rule prevents silent label windows across outages or mixed sequences.  
**What to understand from the result:** Which runs are structurally safe for later label creation and exactly why other runs require human review.

In [ ]:
run_summary = run_summary.set_index(group_keys)
run_summary["duplicated_timestamp_row_count"] = duplicate_counts.reindex(run_summary.index, fill_value=0)
run_summary["backward_timestamp_count"] = backward_counts.reindex(run_summary.index, fill_value=0)
run_summary["large_gap_count"] = large_gap_stats["large_gap_count"].reindex(run_summary.index, fill_value=0).astype(int)
run_summary["maximum_gap_seconds"] = large_gap_stats["maximum_gap_seconds"].reindex(run_summary.index)
run_summary["overlapping_run_count"] = [overlap_counts.get(key, 0) for key in run_summary.index]
run_summary["run_id_machine_count"] = [int(run_machine_counts.loc[key[1]]) for key in run_summary.index]

def review_reasons(row):
    reasons = []
    if row["invalid_timestamp_count"] > 0:
        reasons.append("invalid_timestamp")
    if row["duplicated_timestamp_row_count"] > 0:
        reasons.append("duplicated_timestamp")
    if row["backward_timestamp_count"] > 0:
        reasons.append("timestamp_backward")
    if row["large_gap_count"] > 0:
        reasons.append("large_gap")
    if row["overlapping_run_count"] > 0:
        reasons.append("overlapping_runs")
    if row["run_id_machine_count"] > 1:
        reasons.append("run_id_on_multiple_machines")
    return ";".join(reasons)

run_summary["review_reasons"] = run_summary.apply(review_reasons, axis=1)
run_summary["requires_review"] = run_summary["review_reasons"].ne("")
run_summary["safe_for_future_label_creation"] = ~run_summary["requires_review"]
run_summary = run_summary.reset_index()

issue_columns = ["machine_id", "run_id", "issue_type", "timestamp", "gap_seconds", "severity", "explanation"]
time_audit_df = pd.DataFrame(issue_records, columns=issue_columns)
if not time_audit_df.empty:
    time_audit_df = time_audit_df.sort_values(
        ["machine_id", "run_id", "timestamp", "issue_type"], kind="mergesort"
    ).reset_index(drop=True)

print("Runs considered safe from time-structure checks:")
display(run_summary.loc[run_summary["safe_for_future_label_creation"], ["machine_id", "run_id"]])
print("Runs requiring review:")
display(run_summary.loc[run_summary["requires_review"], ["machine_id", "run_id", "review_reasons"]])

### 9. Save and validate both reports

**What this cell does:** Writes the one-row-per-run summary and every detected issue/warning to the required CSV paths, then reads them back for structural validation.  
**Why it matters for time-series ML:** Persisted, validated reports make later labeling decisions reviewable and reproducible without altering observations.  
**What to understand from the result:** The exact report paths, row counts, and confirmation that all machine/run sequences are represented once.

In [ ]:
run_summary.to_csv(RUN_SUMMARY_PATH, index=False)
time_audit_df.to_csv(TIME_AUDIT_PATH, index=False)

saved_run_summary = pd.read_csv(RUN_SUMMARY_PATH)
saved_time_audit = pd.read_csv(TIME_AUDIT_PATH)

assert len(saved_run_summary) == machine_run_count
assert not saved_run_summary.duplicated(["machine_id", "run_id"]).any()
assert saved_time_audit.columns.tolist() == issue_columns
assert RUN_SUMMARY_PATH.resolve() != INPUT_PATH.resolve()
assert TIME_AUDIT_PATH.resolve() != INPUT_PATH.resolve()

print(f"Created {RUN_SUMMARY_PATH} with {len(saved_run_summary)} run rows.")
print(f"Created {TIME_AUDIT_PATH} with {len(saved_time_audit)} issue/warning rows.")

### 10. Display the final audit summary and verify source integrity

**What this cell does:** Prints all requested headline results and compares the final source checksum with the original fingerprint.  
**Why it matters for time-series ML:** A concise handoff shows whether label engineering can proceed and proves that the audit did not mutate the dataset.  
**What to understand from the result:** Overall safety, issue totals, affected runs, and byte-for-byte source integrity.

In [ ]:
def issue_count(issue_type):
    return int(time_audit_df["issue_type"].eq(issue_type).sum()) if not time_audit_df.empty else 0

safe_runs = run_summary.loc[run_summary["safe_for_future_label_creation"], "run_id"].tolist()
review_runs = run_summary.loc[run_summary["requires_review"], "run_id"].tolist()
input_hash_after = sha256_file(INPUT_PATH)
source_unchanged = input_hash_before == input_hash_after

print("FINAL TIME-STRUCTURE SUMMARY")
print(f"Machines found: {machine_count}")
print(f"Runs found: {machine_run_count}")
print(f"Normal sampling interval: {normal_sampling_interval_seconds:.3f} seconds")
print(f"Duplicated timestamp occurrences: {issue_count('duplicated_timestamp')}")
print(f"Backward timestamps: {issue_count('timestamp_backward')}")
print(f"Large gaps: {issue_count('large_gap')}")
print(f"Overlapping run pairs: {issue_count('overlapping_runs')}")
print(f"Runs safe for future-label creation: {len(safe_runs)} — {safe_runs}")
print(f"Runs requiring review: {len(review_runs)} — {review_runs}")
print(f"Original dataset was not modified: {source_unchanged}")

assert raw_df.shape == original_shape and raw_df.columns.tolist() == original_columns
assert source_unchanged, "The source CSV changed during the audit."
display(run_summary)